## Funnel Analysis
 Understanding Conversion Rates and Drop-off Points
In this notebook we build the core conversion funnel,
calculate drop-off rates at each stage and identify
which categories and brands convert best.

## Import Libraries

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

## Import Data

In [2]:
df = pd.read_csv('data/ecommerce_cleaned.csv')
print(f"Dataset loaded")
print(f"Shape: {df.shape}")


Dataset loaded
Shape: (884964, 13)


## calculate unique users at each funnel stage:

In [3]:
views = df[df['event_type'] == 'view']['user_id'].nunique()
carts = df[df['event_type'] == 'cart']['user_id'].nunique()
purchases = df[df['event_type'] == 'purchase']['user_id'].nunique()

print(f"Users who viewed: {views:,}")
print(f"Users who carted: {carts:,}")
print(f"Users who purchased: {purchases:,}")

Users who viewed: 406,817
Users who carted: 36,948
Users who purchased: 21,304


#### Calculate conversion rates *100

In [4]:
view_to_cart = (carts / views) * 100
cart_to_purchase = (purchases / carts) * 100
overall = (purchases / views) * 100

print(f"View to Cart: {view_to_cart:.2f}%")
print(f"Cart to Purchase: {cart_to_purchase:.2f}%")
print(f"Overall Conversion: {overall:.2f}%")

View to Cart: 9.08%
Cart to Purchase: 57.66%
Overall Conversion: 5.24%


In [5]:
funnel_data = pd.DataFrame({
    'stage': ['View', 'Cart', 'Purchase'],
    'users': [views, carts, purchases]
})

fig = go.Figure(go.Funnel(
    y=funnel_data['stage'],
    x=funnel_data['users'],
    textinfo="value+percent initial",
    marker=dict(color=['steelblue', 'salmon', 'skyblue'])
))

fig.update_layout(
    title=dict(
   text='CUSTOMER CONVERSION FUNNEL<br><sup>View → Cart → Purchase | 406k unique users analysed</sup>',
    font=dict(size=20)
    )
)
fig.show()

## Key Insights:
 Drop 1 → View to Cart (biggest problem)

- 406,817 → 36,948
- 91% of users who viewed never added to cart
This is where you're losing the most customers

Drop 2 → Cart to Purchase (smaller problem)

- 36,948 → 21,304
- 57.66% of users who added to cart actually purchased
This is actually quite good!

The big story:
The business doesn't have a checkout problem, it has a product discovery and engagement problem. Getting people from browsing to cart is where the revenue is being lost.
## Business  Implication:
Improving view-to-cart conversion by just 2% would add approximately 8,000 more buyers, representing significant revenue uplift

## Category Level Conversion 

In [35]:
views_cat = df[df['event_type'] == 'view'].groupby(
    'category_code')['user_id'].nunique()

purchases_cat = df[df['event_type'] == 'purchase'].groupby(
    'category_code')['user_id'].nunique()

# Combine into dataframe
category_funnel = pd.DataFrame({
    'views': views_cat,
    'purchases': purchases_cat
}).fillna(0)

category_funnel['conversion_rate'] = (
    category_funnel['purchases'] / 
    category_funnel['views'] * 100
).round(2)

# Top 10 by conversion rate
top_converting = category_funnel[
    category_funnel['views'] > 100
].sort_values('conversion_rate', ascending=False).head(10)

print(top_converting)

                                   views  purchases  conversion_rate
category_code                                                       
computers.peripherals.camera        2070      328.0            15.85
computers.components.videocards    29032     3980.0            13.71
computers.peripherals.scanner        819       94.0            11.48
electronics.video.projector          725       63.0             8.69
computers.ebooks                    1553      132.0             8.50
computers.components.power_supply   3605      296.0             8.21
stationery.stapler                   375       30.0             8.00
computers.peripherals.printer      19732     1519.0             7.70
computers.components.motherboard   10087      749.0             7.43
stationery.cartrige                20905     1475.0             7.06


In [49]:
fig = px.bar(
    top_converting.reset_index().sort_values('conversion_rate', ascending=True),
    x='conversion_rate',
    y='category_code',
    orientation='h',
    title='TOP 10 CATEGORIES BY CONVERSION RATE<br><sup>Based on unique users who viewed vs purchased</sup>',
    labels={
        'conversion_rate': 'Conversion Rate (%)',
        'category_code': 'Category'
    },
    color='conversion_rate',
    color_continuous_scale='Blues',
    text='conversion_rate'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(
    coloraxis_showscale=False,
    width=900,
    height=550,
    margin=dict(l=250, r=200)
)
fig.show()

## Key Insights:
- Computers.peripherals.camera leads with a 15.85% conversion 
rate but low traffic at 2,070 views.
- Computers.components.videocards at 13.71% with 29,032 views represents the greater revenue 
opportunity, combining strong conversion with high traffic volume.

## Business Implication

Marketing investment should prioritise videocards given its 
superior combination of high traffic and strong conversion. 
Cameras and scanners hold high conversion potential but need 
targeted campaigns to grow their audience and unlock greater 
revenue impact.

## Brand Conversion Analysis 
Calculate conversion rate by brand

In [37]:
views_brand = df[df['event_type'] == 'view'].groupby(
    'brand')['user_id'].nunique()

purchases_brand = df[df['event_type'] == 'purchase'].groupby(
    'brand')['user_id'].nunique()

# Combine into dataframe
brand_funnel = pd.DataFrame({
    'views': views_brand,
    'purchases': purchases_brand
}).fillna(0)

brand_funnel['conversion_rate'] = (
    brand_funnel['purchases'] / 
    brand_funnel['views'] * 100
).round(2)

# Filter brands with meaningful volume
top_converting_brands = brand_funnel[
    brand_funnel['views'] > 500
].sort_values('conversion_rate', ascending=False).head(10)

print(top_converting_brands)

             views  purchases  conversion_rate
brand                                         
sapphire      3636      578.0            15.90
steelseries    708      107.0            15.11
msi           7716     1125.0            14.58
nv-print      1860      225.0            12.10
gigabyte      9425     1128.0            11.97
powercolor    1732      187.0            10.80
pantum        1454      156.0            10.73
brother       1130      110.0             9.73
seagate        580       52.0             8.97
sandisk        689       60.0             8.71


In [44]:
fig = px.bar(
    top_converting_brands.reset_index().sort_values(
        'conversion_rate', ascending=True),
    x='conversion_rate',
    y='brand',
    orientation='h',
    title='TOP 10 BRANDS BY CONVERSION RATE<br><sup>Based on unique users who viewed vs purchased</sup>',
    labels={
        'conversion_rate': 'Conversion Rate (%)',
        'brand': 'Brand'
    },
    color='conversion_rate',
    color_continuous_scale='Blues',
    text='conversion_rate'
)
fig.update_traces(
    texttemplate='%{text:.1f}%', 
    textposition='outside'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## Key Insights: 
- Sapphire leads all brands with a 15.90% conversion rate, 
  a GPU brand consistent with videocards being the top 
  converting category, confirming strong PC components demand

- Steelseries at 15.11% is a standout finding, as a gaming 
  peripherals brand, its high conversion suggests a strong 
  gaming audience segment worth targeting with dedicated 
  campaigns

- MSI and Gigabyte both appear in the top 5, confirming 
  that PC component brands as a whole drive the strongest 
  conversion performance on the platform

- However, conversion rate alone does not tell the full story. 
  Gigabyte with 9,425 views and 11.97% conversion represents 
  a greater revenue opportunity than Sapphire (3,636 views) 
  despite Sapphire's higher rate. High traffic combined with 
  strong conversion is where the real revenue impact lies

## Business Implication

- Prioritise marketing partnerships with Gigabyte and MSI. 
  They offer the best combination of high traffic volume and 
  strong conversion rates
- Develop a dedicated gaming audience strategy targeting 
  Steelseries and peripherals buyers,  this segment converts 
  well and may be currently underleveraged

#### Average Price at Each Funnel Stage 

In [40]:
price_by_stage = df.groupby(
    'event_type'
)['price'].mean().round(2).reset_index()
price_by_stage.columns = ['event_type', 'avg_price']

print(price_by_stage)

  event_type  avg_price
0       cart     159.65
1   purchase     137.24
2       view     145.84


In [54]:
fig = px.bar(
    price_by_stage,
    x='event_type',
    y='avg_price',
    title='AVERAGE PRICE BY FUNNEL STAGE<br><sup>Comparing price behaviour across view, cart and purchase</sup>', 
    labels={'avg_price': 'Average Price (£)',
      'event_type': 'Event Type'},
    color='event_type',
    color_discrete_map={
        'view': 'steelblue',
        'cart': 'salmon',
        'purchase': 'skyblue'
    },
    text='avg_price'
)
fig.update_traces(
    texttemplate='£%{text:.2f}',
    textposition='outside'
)
fig.show()

## Key Insights: 
- The £22.41 gap between average cart price (£159.65) and 
purchase price (£137.24) reveals a 14% price sensitivity threshold. Customers browse premium items but downgrade  at checkout
- Viewed items (£145.84) sitting between cart and purchase 
  prices confirms customers actively seek premium products 
  but self-select lower priced items when committing to buy

## Business Implication

- A targeted 10-15% discount on abandoned cart items above 
  £150 could bridge this gap and recover significant lost revenue

- Introducing Buy Now Pay Later (BNPL) options for high value 
  items could reduce price hesitation and boost conversion

## Summary: Key Insights & Strategic Recommendations

### Key Insights

❖ The platform has a product discovery problem, not a 
checkout problem. Only 9.08% of viewers add to cart 
but 57.66% of cart users complete purchase. The 
critical drop-off happens before checkout.

❖ PC components and electronics dominate both traffic 
and conversion. Videocards lead volume at 116,712 
events while cameras lead conversion at 15.85% , 
confirming a specialist technical audience.

❖ Gigabyte represents the greatest revenue opportunity 
combining high traffic (9,425 views) with strong 
conversion (11.97%)  outweighing higher converting 
but lower traffic brands like Sapphire.

❖ A 14% price sensitivity threshold exists between 
cart (£159.65) and purchase (£137.24). Customers 
browse premium but buy affordable.

### Strategic Recommendations

❖ Prioritise view-to-cart optimisation through improved 
product pages, urgency signals and personalised 
recommendations,  a 2% improvement adds 8,000 buyers.

❖ Focus marketing partnerships on Gigabyte and MSI 
for maximum revenue impact.

❖ Introduce BNPL options for items above £150 to 
bridge the price sensitivity gap.

❖ Develop dedicated gaming audience campaigns targeting 
Steelseries buyers. High conversion, underleveraged.

❖  Concentrate promotional spend between 10am and 6pm 
where traffic peaks at 40k-45k events. Evening spend 
yields significantly lower returns.